# AHR PAS-B Screening — Approved-Drug Repurposing (hierarchical)

Protocol validated by redocking (best pose RMSD **0.49 Å** vs crystal
indirubin in 7ZUB chain D) — see `research/AHR_redocking_result.md`.

**Hierarchical virtual screening** (fast → accurate):
1. **Stage A** — Vina-only breadth screen of the ≤34-heavy-atom subset
   of 589 approved drugs (`--cnn_scoring none --exhaustiveness 2`),
   chunked + parallel with per-chunk checkpoints.
2. **Stage B** — CNN refinement (`--cnn fast --cnn_scoring rescore`,
   `--exhaustiveness 12`) of the **top 30** Vina hits.
3. **Stage C** — rank by CNN affinity, check pocket contacts vs
   His337 / Gln383 / Tyr322.

The free Colab T4 has only 2 vCPUs and GNINA's Monte-Carlo sampling is
CPU-bound, so stage A deliberately uses no CNN scoring and a compact
ligand filter; accuracy comes from the CNN refinement of the shortlist.

Library: 589 approved drugs, MaxMin diversity pick from 2,838 dockable
ChEMBL `max_phase=4` molecules, protonated pH 7.4, ETKDGv3 3D.

**Run on GPU:** Runtime → T4 GPU, then Run all.

In [ ]:
# 1. GNINA + dependencies
!wget -q https://github.com/gnina/gnina/releases/download/v1.3.3/gnina.cuda12.8.static -O gnina
!chmod +x gnina
!apt-get -qq update && apt-get -qq install -y openbabel 2>/dev/null | tail -1
!pip install -q rdkit biopython pandas
!./gnina --version

In [ ]:
# 2. Receptor prep - identical to the validated redocking
!wget -q https://files.rcsb.org/download/7ZUB.pdb

from Bio.PDB import PDBParser, PDBIO, Select
s = PDBParser(QUIET=True).get_structure('7ZUB', '7ZUB.pdb')
class ProtOnly(Select):
    def accept_chain(self, c): return c.id == 'D'
    def accept_residue(self, r): return r.id[0] == ' '
class LigOnly(Select):
    def accept_residue(self, r): return r.get_resname() == 'JY6'
io = PDBIO(); io.set_structure(s)
io.save('receptor.pdb', ProtOnly())
io.save('indirubin.pdb', LigOnly())
!obabel receptor.pdb -xr -h -p 7.4 -O receptor_prep.pdb 2>/dev/null
!echo "receptor_prep: $(grep -c '^ATOM' receptor_prep.pdb) atoms"

In [ ]:
# 3. Fetch the screening library (secret gist; ChEMBL approved drugs)
!wget -q https://gist.githubusercontent.com/skadlem/3b89119ebd49a7cfa0dd5ad6d64ea903/raw/library_batch1.sdf -O library.sdf
from rdkit import Chem
n = sum(1 for _ in Chem.SDMolSupplier('library.sdf', removeHs=True, sanitize=False))
print(f'library.sdf: {n} compounds')

In [ ]:
import subprocess, time, os
from concurrent.futures import ThreadPoolExecutor
from rdkit import Chem

# Filter to <= 34 heavy atoms: the AHR PAS-B pocket is compact (indirubin = 20 HA),
# so larger molecules rarely fit and dominate runtime on this 2-vCPU instance.
sup = Chem.SDMolSupplier('library.sdf', removeHs=True, sanitize=False)
keep = [m for m in sup if m is not None and m.GetNumHeavyAtoms() <= 34]
w = Chem.SDWriter('library_filtered.sdf')
for m in keep:
    w.write(m)
w.close()
print(f'{len(keep)}/589 pass <=34 heavy atoms', flush=True)

CH, NPROC = 75, 2
nchunks = (len(keep) + CH - 1) // CH
for k in range(nchunks):
    w = Chem.SDWriter(f'lib_{k:02d}.sdf')
    for m in keep[k * CH:(k + 1) * CH]:
        w.write(m)
    w.close()
print(f'{len(keep)} -> {nchunks} chunks of ~{CH}', flush=True)

def run(k):
    out = f'stageA_{k:02d}.sdf.gz'
    env = {**os.environ, 'OMP_NUM_THREADS': '1'}
    t = time.time()
    r = subprocess.run(['./gnina', '-r', 'receptor_prep.pdb', '-l', f'lib_{k:02d}.sdf',
                        '--autobox_ligand', 'indirubin.pdb', '--autobox_add', '5',
                        '--cnn_scoring', 'none', '--exhaustiveness', '2', '--num_modes', '1',
                        '--cpu', '1', '-o', out],
                       capture_output=True, text=True, timeout=5400, env=env)
    return k, r.returncode, time.time() - t, out, (r.stderr or '')[-200:]

t0 = time.time(); ok = 0
with ThreadPoolExecutor(max_workers=NPROC) as ex:
    for k, rc, dt, out, err in ex.map(run, range(nchunks)):
        if rc != 0:
            print(f'chunk {k}: FAILED rc={rc}\n{err}', flush=True)
        else:
            ok += 1
            el = (time.time() - t0) / 60
            print(f'chunk {k}: ok {dt/60:.1f}min | {ok}/{nchunks} | elapsed {el:.1f}min | ETA {el/ok*(nchunks-ok):.1f}min', flush=True)
print(f'STAGE A done: {ok}/{nchunks} chunks in {(time.time()-t0)/60:.1f} min', flush=True)


In [ ]:
# 5. STAGE B - CNN refinement of the top Vina hits
# Two environment quirks, both worked around here:
#  - with --cnn_scoring none GNINA writes the Vina score as <minimizedAffinity>,
#    not <Affinity>;
#  - RDKit's SDMolSupplier mis-reads GNINA's .sdf.gz in this Colab image (it
#    sees the whole file as one unparseable record), so every file is
#    decompressed to plain .sdf first. A checkpoint truncated by an interrupted
#    run fails at gzip-open and is skipped instead of aborting the stage.
from rdkit import Chem
import gzip, glob, os, subprocess, time, pandas as pd

def vina(m):
    for k in ('minimizedAffinity', 'Affinity'):
        if m.HasProp(k):
            try:
                return float(m.GetProp(k))
            except ValueError:
                pass
    return None

def read_stageA(f):
    """Decompress + parse one checkpoint. A checkpoint truncated by an
    interrupted run unzips to a short SDF; we keep whatever parsed before the
    truncation point instead of letting the supplier exception kill the stage."""
    out = f.replace('.sdf.gz', '.sdf')
    try:
        with gzip.open(f, 'rt', errors='replace') as fh:
            open(out, 'w').write(fh.read())
    except Exception as e:
        print(f'{f}: corrupt gzip, skipped ({e})', flush=True)
        return []
    mols = []
    try:
        for m in Chem.SDMolSupplier(out, removeHs=True, sanitize=False):
            if m is not None:
                mols.append(m)
    except Exception as e:
        print(f'{f}: truncated SDF, kept {len(mols)} usable poses', flush=True)
    return mols

best = {}
for f in sorted(glob.glob('stageA_*.sdf.gz')):
    mols = read_stageA(f)
    if not mols:
        continue
    n = 0
    for m in mols:
        if m is None:
            continue
        aff = vina(m)
        if aff is None:
            continue
        cid = m.GetProp('_Name') or (m.GetProp('chembl_id') if m.HasProp('chembl_id') else '')
        nm = m.GetProp('name') if m.HasProp('name') else cid
        if not cid:
            continue
        n += 1
        if cid not in best or aff < best[cid][0]:
            best[cid] = (aff, nm)
    print(f'{f}: {n} scored', flush=True)

print(f'{len(best)} unique compounds with a Vina score in stage A', flush=True)
pd.DataFrame([(cid, aff, nm) for cid, (aff, nm) in best.items()],
             columns=['id', 'vina_affinity', 'name']).sort_values(
             'vina_affinity').to_csv('stageA_results.csv', index=False)
print('wrote stageA_results.csv', flush=True)

lib = {}
for m in Chem.SDMolSupplier('library.sdf', removeHs=True, sanitize=False):
    if m is not None:
        lib[m.GetProp('_Name')] = m
print(f'library index: {len(lib)} structures', flush=True)

# If the stage A checkpoints are absent (e.g. a fresh runtime after a
# disconnect), fall back to the top-15 Vina hits recorded in
# research/screen_stageA_top30.md and go straight to CNN refinement.
KNOWN_TOP15 = ['CHEMBL1201174', 'CHEMBL535650', 'CHEMBL3989973', 'CHEMBL3545253',
               'CHEMBL1173055', 'CHEMBL295124', 'CHEMBL2105743', 'CHEMBL1201753',
               'CHEMBL2107387', 'CHEMBL1475', 'CHEMBL1089318', 'CHEMBL1709',
               'CHEMBL1200374', 'CHEMBL1301', 'CHEMBL1218']

TOPN = 30
ranked = sorted(best.items(), key=lambda kv: kv[1][0])[:TOPN]
if not ranked:
    print('no stage A checkpoints found - using recorded top-15 Vina hits', flush=True)
    ranked = [(cid, (0.0, '')) for cid in KNOWN_TOP15]
w = Chem.SDWriter('shortlist.sdf')
n_ok = 0
for cid, (aff, nm) in ranked:
    if cid in lib:
        w.write(lib[cid])
        n_ok += 1
w.close()
print(f'shortlist: {n_ok}/{len(ranked)} top Vina hits -> shortlist.sdf', flush=True)
for cid, (aff, nm) in ranked[:15]:
    print(f'  {cid}  {nm[:26]:<26} vina={aff:.2f}', flush=True)

t = time.time()
r = subprocess.run(['./gnina', '-r', 'receptor_prep.pdb', '-l', 'shortlist.sdf',
                    '--autobox_ligand', 'indirubin.pdb', '--autobox_add', '5',
                    '--cnn', 'fast', '--cnn_scoring', 'rescore',
                    '--exhaustiveness', '12', '--num_modes', '3',
                    '-o', 'refined.sdf.gz'],
                   capture_output=True, text=True, timeout=5400)
print(f'STAGE B rc={r.returncode} in {(time.time()-t)/60:.1f} min', flush=True)
if r.returncode != 0:
    print((r.stderr or '')[-500:], flush=True)


In [ ]:
# 7. STAGE C - rank refined hits + pocket contacts
# His337 is the closest residue to the co-bound indirubin (2.8 A);
# Gln383 / Tyr322 / Ser336 are the only plausible H-bond partners.
# NB: CNN scores are near-degenerate across binding modes (see the redocking
# result - the rank-1 pose was a 6.55 A decoy), so we report the BEST pose per
# compound, not the first.
from rdkit import Chem
from Bio.PDB import PDBParser
import numpy as np, pandas as pd

KEY = {'HIS337', 'GLN383', 'TYR322', 'LEU308', 'ILE325', 'LEU353', 'PHE351', 'PHE287'}

rec = PDBParser(QUIET=True).get_structure('R', 'receptor_prep.pdb')[0]
res_atoms = [(res.get_resname() + str(res.id[1]),
              np.array([a.coord for a in res.get_atoms()])) for res in rec.get_residues()]

def g(m, k):
    return float(m.GetProp(k)) if m.HasProp(k) else None

# RDKit mis-reads GNINA's .sdf.gz in this image - decompress first.
import gzip
with gzip.open('refined.sdf.gz', 'rt', errors='replace') as fh:
    open('refined.sdf', 'w').write(fh.read())

best_per = {}
for m in Chem.SDMolSupplier('refined.sdf', removeHs=True, sanitize=False):
    if m is None:
        continue
    cid = m.GetProp('_Name')
    px = m.GetConformer().GetPositions()
    contacts = []
    for name, xyz in res_atoms:
        d = float(np.linalg.norm(xyz - px, axis=1).min())
        if d <= 4.5:
            contacts.append((name, round(d, 1)))
    contacts.sort(key=lambda t: t[1])
    cnn_aff = g(m, 'CNNaffinity')
    info = {
        'id': cid,
        'name': m.GetProp('name') if m.HasProp('name') else cid,
        'cnn_aff': cnn_aff,
        'cnn_pose': g(m, 'CNNscore'),
        'vina': g(m, 'Affinity'),
        'n_contacts': len(contacts),
        'contacts': contacts,
    }
    key = info['cnn_aff'] if info['cnn_aff'] is not None else info['vina']
    if cid not in best_per or key > best_per[cid]['_rankkey']:
        info['_rankkey'] = key
        best_per[cid] = info

ranked = sorted(best_per.values(), key=lambda r: -r['_rankkey'])
print(f'{len(ranked)} refined compounds\n')
hdr = f"{'CHEMBL':<10}{'name':<22}{'CNNaff':>7}{'CNNpose':>8}{'vina':>7}{'#cont':>6}  key-contacts"
print(hdr); print('-' * len(hdr))
for r in ranked[:20]:
    kc = [n for n, d in r['contacts'] if n in KEY]
    ca = f"{r['cnn_aff']:>7.2f}" if r['cnn_aff'] is not None else '     --'
    cp = f"{r['cnn_pose']:>8.3f}" if r['cnn_pose'] is not None else '       --'
    print(f"{r['id']:<10}{r['name'][:21]:<22}{ca}{cp}"
          f"{r['vina']:>7.2f}{r['n_contacts']:>6}  {' '.join(kc[:6])}")

df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('contacts', '_rankkey')}
                   for r in ranked])
df['top_contacts'] = [' '.join(f'{n}({d})' for n, d in r['contacts'][:8]) for r in ranked]
df.to_csv('screen_results.csv', index=False)
print('\nwrote screen_results.csv')
